# SINDYc 5 · Why does holding win, and can the model learn it? (exploratory)

Always hold/focus wins more battles than always advance/focus (70.2% against 67.0% in the
planner test), but the SINDYc model can't see it, so the model planner keeps choosing advance.
Three steps:

1. **What differs** between holding and advancing from the same start, at the first contact?
2. **What is the model missing?** Give its rollouts one real quantity at a time (shape,
   distance, reach) and see which one makes it predict the hold advantage.
3. **Teach the model** that quantity: reach as its own SINDYc state.
4. **A cleaner test, and better formation numbers.** Section 2 leaks: the real reach also
   carries the real deaths. Redone with the model's own deaths, and with new formation numbers.

Data: 2,000 starts, each played with always hold/focus and always advance/focus
(`scripts/play_hold_advance_pairs.py`). Model: the sparse SINDYc model with hold split by contact
(`model_split.pkl`, fitted on exp06 train with `exclude=ALIVE_CONTROL_TERMS + ("close*hold",)`).

In [ ]:
import pickle
import warnings

import numpy as np
import pandas as pd

from sindy_wm.models.intent_model import (
    CTRL,
    DT,
    REACH_COLS,
    SHAPE,
    STATE,
    _carry,
    add_controls,
    add_rates,
    control_vector,
    library,
    r2,
    share,
    stlsq,
)
from sindy_wm.paths import DATA_DIR

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
PROCESSED = DATA_DIR / "processed" / "exp06_intents"
pairs = pd.read_parquet(PROCESSED / "pairs_hold_advance.parquet").sort_values(["seed", "movement", "step"])
model = pickle.load(open(PROCESSED / "model_split.pkl", "rb"))
won = pairs.groupby(["seed", "movement"])["won"].first().unstack()
print(f"{len(won)} starts. Blue wins: hold {won['hold'].mean():.1%}, advance {won['advance'].mean():.1%}")
print(
    f"hold wins where advance loses: {(won['hold'] & ~won['advance']).sum()}, "
    f"the reverse: {(~won['hold'] & won['advance']).sum()}"
)

## 1 · What differs at first contact?

First contact = the first step where any unit of either side has an enemy within reach. For
each start and each intent: the formations at that moment, how many units on each side are in
reach during the first 2 s, and how many weapons are ready.

In [ ]:
records = []
for (seed, movement), g in pairs.groupby(["seed", "movement"]):
    g = g.reset_index(drop=True)
    k = np.flatnonzero((g["reach_blue"] + g["reach_red"]) > 0)[0]
    first2 = g.iloc[k : k + 16]
    margin = g["health_blue"] - g["health_red"]
    records.append(
        {
            "seed": seed,
            "movement": movement,
            "final margin": margin.iloc[-1],
            "contact time (s)": g["t"].iloc[k],
            "distance at contact": g["distance"].iloc[k],
            "depth blue": g["depth_blue"].iloc[k],
            "depth red": g["depth_red"].iloc[k],
            "width blue": g["width_blue"].iloc[k],
            "width red": g["width_red"].iloc[k],
            "weapons ready blue": g["ready_blue"].iloc[k],
            "weapons ready red": g["ready_red"].iloc[k],
            "Blue units in reach, first 2 s": 10 * first2["reach_blue"].mean(),
            "Red units in reach, first 2 s": 10 * first2["reach_red"].mean(),
            "margin change, first 4 s of contact": margin.iloc[min(k + 32, len(g) - 1)] - margin.iloc[k],
        }
    )
contact = pd.DataFrame(records)
contact["reach advantage, first 2 s"] = (
    contact["Blue units in reach, first 2 s"] - contact["Red units in reach, first 2 s"]
)
hold, adv = (
    contact[contact["movement"] == m].set_index("seed").drop(columns="movement") for m in ("hold", "advance")
)
diff = hold - adv
pd.DataFrame(
    {
        "hold": hold.mean(),
        "advance": adv.mean(),
        "hold - advance": diff.mean(),
        "share of starts where hold is higher": (diff > 1e-9).mean(),
        "corr. with the difference in final margin": diff.corrwith(diff["final margin"]),
    }
).round(3)

**What the numbers say:** fill in. Questions to ask of the table: does Red arrive more spread out
when Blue holds? Which side's formation changes? Which measure goes with the difference in
outcome? How much of the final difference is made in the first 4 s of contact?

## 2 · What is the model missing?

From the start of each battle, the model predicts the margin after 12 s under each intent. The
predicted difference (hold - advance) is compared with the real one. Then the rollouts are given
one real quantity over the horizon, taken from the battle played with that intent.

In [ ]:
H = 96  # 12 s
seeds = np.sort(pairs["seed"].unique())


def trajectories(movement, cols):
    g = pairs[pairs["movement"] == movement]
    out = np.full((len(seeds), H + 1, len(cols)), np.nan)
    for j, (_, e) in enumerate(g.groupby("seed")):
        n = min(len(e), H + 1)
        out[j, :n] = e[cols].to_numpy()[:n]
    return out


truth = {
    m: {k: trajectories(m, c) for k, c in (("state", STATE), ("shape", SHAPE), ("reach", REACH_COLS))}
    for m in ("hold", "advance")
}


def last_margin(Y):
    last = np.array([y[~np.isnan(y[:, 0])][-1] for y in Y])
    return last[:, 0] - last[:, 1]


real = last_margin(truth["hold"]["state"]) - last_margin(truth["advance"]["state"])
X0, S0, R0 = (truth["hold"][k][:, 0] for k in ("state", "shape", "reach"))
differs = np.abs(real) > 0.02


def controls(movement):
    return np.repeat(np.repeat(control_vector(movement, "focus")[None], H, 0)[None], len(seeds), 0)


def compare(predict):
    p = predict("hold") - predict("advance")
    return {
        "predicted hold - advance (mean)": p.mean(),
        "real (mean)": real.mean(),
        "correlation with real": np.corrcoef(p, real)[0, 1],
        "sign right where they differ": np.mean(np.sign(p[differs]) == np.sign(real[differs])),
        "predicts no difference where they differ": np.mean(np.abs(p[differs]) < 1e-9),
    }


def rollout(**kw):
    def predict(movement):
        t = {k: truth[movement][v][..., sl] for k, (v, sl) in kw.get("truth", {}).items()}
        Y = model.simulate(X0, S0, controls(movement), reach0=R0, truth=t, shape=kw.get("shape", "hold"))
        return Y[:, -1, 0] - Y[:, -1, 1]

    return predict


print(
    f"starts where hold and advance end differently (by > 0.02): {differs.mean():.0%}; "
    f"hold better in {np.mean(real > 0.02):.0%}"
)
pd.DataFrame(
    {
        "model alone (shape held)": compare(rollout()),
        "model, shape from its equations": compare(rollout(shape="model")),
        "+ real shape": compare(rollout(truth={"shape": ("shape", slice(None))})),
        "+ real distance": compare(rollout(truth={"distance": ("state", 4)})),
        "+ real units in reach": compare(rollout(truth={"reach": ("reach", slice(None))})),
    }
).T.round(3)

**Careful:** the real units in reach also carry the real *deaths* (a dead unit can't be in
reach), so the last row partly hands the model the outcome. Section 4 redoes this test cleanly.

## 3 · Reach as its own SINDYc state

If the model needs the real reach, give reach an equation. Candidates per side, each times a
movement (advance, hold, withdraw):

- `(E-R)`: relax towards the geometric equilibrium E = alive x reach curve
- `(A-R)*contact`: units out of reach joining the fight once there is contact
- `R`: units leaving reach

and a second library that adds `(A-R)*contact*movement` times own depth and own width (a wide,
shallow line should bring units into reach faster than a column).

In [ ]:
train = add_rates(
    add_controls(pd.read_parquet(PROCESSED / "train.parquet")), cols=tuple(STATE + SHAPE + REACH_COLS)
)
rows = train[(train["alive_blue"] > 0) & (train["alive_red"] > 0)]
SIDES = (("blue", 0, 1), ("red", 1, 0))


def reach_library(X, U, S, R, shape_terms=False):
    db, dr, wb, wr = S.T
    E = (
        X[:, 2] * share(X[:, 4], db, dr, wb, wr, *model.P["share_blue"]),
        X[:, 3] * share(X[:, 4], dr, db, wr, wb, *model.P["share_red"]),
    )
    out = {}
    for side, j, o in SIDES:
        contact = 1 - np.exp(-10 * (R[:, j] + R[:, o]))
        F = {}
        for name, u in (("adv", U[:, 0]), ("hold", U[:, 1]), ("wd", U[:, 2])):
            F[f"(E-R)*{name}"] = (E[j] - R[:, j]) * u
            F[f"(A-R)*contact*{name}"] = (X[:, 2 + j] - R[:, j]) * contact * u
            F[f"R*{name}"] = R[:, j] * u
            if shape_terms and name != "wd":
                F[f"(A-R)*contact*{name}*depth"] = F[f"(A-R)*contact*{name}"] * S[:, j]
                F[f"(A-R)*contact*{name}*width"] = F[f"(A-R)*contact*{name}"] * S[:, 2 + j]
        out[side] = F
    return out


def fit_reach(shape_terms):
    L = reach_library(
        *(rows[c].to_numpy() for c in (STATE, CTRL, SHAPE, REACH_COLS)), shape_terms=shape_terms
    )
    coef = {}
    for side, _, _ in SIDES:
        T, y = np.column_stack(list(L[side].values())), rows[f"d_reach_{side}"].to_numpy()
        coef[side] = stlsq(T, y, 0.05)
        terms = " ".join(f"{c:+.3f} {n}" for n, c in zip(L[side], coef[side], strict=True) if c)
        print(f"  (reach_{side})' = {terms}   [R² {r2(y, T @ coef[side]):.2f}]")
    return coef


def reach_rollout(coef, shape_terms, real_shape=False):
    def predict(movement):
        X, R, U = X0.copy(), R0.copy(), controls(movement)
        Ch = np.stack([model.coef["health_blue"], model.coef["health_red"]])
        for n in range(H):
            u, S = U[:, n], (_carry(truth[movement]["shape"], n) if real_shape else S0)
            F = library(X, u, S, model.P, R, model.ctrl)
            theta = np.column_stack([F[k] for k in model.names])
            L = reach_library(X, u, S, R, shape_terms)
            dR = np.column_stack([np.column_stack(list(L[s].values())) @ coef[s] for s, _, _ in SIDES])
            Hn = np.maximum(X[:, :2] + DT * theta @ Ch.T, 0)
            ratio = np.divide(Hn, X[:, :2], out=np.zeros_like(Hn), where=X[:, :2] > 1e-9)
            A = X[:, 2:4] * np.column_stack(
                [ratio[:, 0] ** model._p_of("blue", u), ratio[:, 1] ** model._p_of("red", u)]
            )
            R = np.clip(R + DT * dR, 0, A)
            X = np.column_stack([Hn, A, np.maximum(X[:, 4] + DT * theta @ model.coef["distance"], 0)])
        return X[:, 0] - X[:, 1]

    return predict


results = {}
for shape_terms in (False, True):
    label = "reach state" + (" + shape terms" if shape_terms else "")
    print(label)
    coef = fit_reach(shape_terms)
    results[label] = compare(reach_rollout(coef, shape_terms))
    results[label + ", real shape"] = compare(reach_rollout(coef, shape_terms, real_shape=True))
pd.DataFrame(results).T.round(3)

**What the numbers say:** fill in.

## 4 · A cleaner test, and better formation numbers

The model now keeps its **own** units alive; only the **share** of alive units in reach comes from
outside, at every step of the horizon. Sources for that share:

- the hand-fitted reach curve, from the real distance and shape
- a flexible map (gradient boosting, fitted on 2,000 exp06 training battles) from four sets of
  formation numbers: **A** depth and width (today's), **B** + how far the front unit is ahead of
  its centre, **C** + the share of units within 1 and 2 of the front unit, **D** deciles of the
  unit positions along the line between the teams (nearly the whole formation)
- the real share (the best any formation number can do)

First: how well does each set predict units in reach, at the same moment (exp06 validation)?

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor  # noqa: E402

geometry_cols = [f"{n}_{s}" for n in ("depth", "width") for s in ("blue", "red")]


def with_formation(states_file, formation_file):
    st = pd.read_parquet(PROCESSED / states_file).reset_index()
    f = pd.read_parquet(PROCESSED / formation_file).drop(columns=geometry_cols)
    d = st.merge(f, on=["episode_idx", "step"])
    return d[(d["alive_blue"] > 0) & (d["alive_red"] > 0)]


f_train = with_formation("train.parquet", "formation_train2000.parquet")
f_val = with_formation("val.parquet", "formation_val.parquet")


def feature_sets(side):
    o = "red" if side == "blue" else "blue"
    A = ["distance", f"depth_{side}", f"depth_{o}", f"width_{side}", f"width_{o}"]
    B = A + [f"front_{side}", f"front_{o}"]
    C = B + [f"near1_{side}", f"near2_{side}", f"near1_{o}", f"near2_{o}", f"q90_{side}", f"q90_{o}"]
    D = ["distance", f"width_{side}", f"width_{o}"] + [f"dec{k}_{x}" for k in range(1, 11) for x in (side, o)]
    return {
        "A depth, width": A,
        "B + front unit": B,
        "C + units near the front": C,
        "D deciles (upper bound)": D,
    }


maps, fit_table = {}, {}
for side, o in (("blue", "red"), ("red", "blue")):
    s_hand = share(
        f_val["distance"].values,
        f_val[f"depth_{side}"].values,
        f_val[f"depth_{o}"].values,
        f_val[f"width_{side}"].values,
        f_val[f"width_{o}"].values,
        *model.P[f"share_{side}"],
    )
    fit_table[("hand-fitted reach curve", side)] = r2(
        f_val[f"reach_{side}"].values, f_val[f"alive_{side}"].values * s_hand
    )
    sample = f_train.sample(150_000, random_state=0)
    for name, cols in feature_sets(side).items():
        m = HistGradientBoostingRegressor(max_iter=300, max_depth=6, learning_rate=0.1)
        m.fit(sample[cols], sample[f"reach_{side}"] / sample[f"alive_{side}"])
        maps[(side, name)] = (m, cols)
        pred = f_val[f"alive_{side}"].values * np.clip(m.predict(f_val[cols]), 0, 1)
        fit_table[(name, side)] = r2(f_val[f"reach_{side}"].values, pred)
pd.Series(fit_table).unstack().round(3).rename_axis("units in reach, R² (validation)")

Then the decision test: the paired battles, with the share in reach from each source.

In [ ]:
form_pairs = pd.read_parquet(PROCESSED / "pairs_hold_advance_formation.parquet").drop(columns=geometry_cols)
both = pairs.merge(form_pairs, on=["seed", "movement", "step"], how="left").sort_values(
    ["seed", "movement", "step"]
)


def share_over_horizon(movement, source):
    g = both[both["movement"] == movement]
    sh = {}
    for side, o in (("blue", "red"), ("red", "blue")):
        if source == "real share in reach":
            sh[side] = (g[f"reach_{side}"] / g[f"alive_{side}"]).replace(np.inf, np.nan).values
        elif source == "hand-fitted reach curve":
            sh[side] = share(
                g["distance"].values,
                g[f"depth_{side}"].values,
                g[f"depth_{o}"].values,
                g[f"width_{side}"].values,
                g[f"width_{o}"].values,
                *model.P[f"share_{side}"],
            )
        else:
            m, cols = maps[(side, source)]
            X = g[cols].to_numpy(float)
            ok = ~np.isnan(X).any(axis=1)
            sh[side] = np.full(len(g), np.nan)
            sh[side][ok] = np.clip(m.predict(X[ok]), 0, 1)
    g = g.assign(sb=sh["blue"], sr=sh["red"])
    out = np.full((len(seeds), H + 1, 2), np.nan)
    for j, (_, e) in enumerate(g.groupby("seed")):
        n = min(len(e), H + 1)
        out[j, :n] = e[["sb", "sr"]].to_numpy()[:n]
    return out


def share_rollout(source):
    def predict(movement):
        S_share, U = share_over_horizon(movement, source), controls(movement)
        X = X0.astype(float).copy()
        Ch = np.stack([model.coef["health_blue"], model.coef["health_red"]])
        for n in range(H):
            u = U[:, n]
            R = X[:, 2:4] * np.nan_to_num(_carry(S_share, n))  # the model's own alive units
            F = library(X, u, S0, model.P, R, model.ctrl)
            theta = np.column_stack([F[k] for k in model.names])
            Hn = np.maximum(X[:, :2] + DT * theta @ Ch.T, 0)
            ratio = np.divide(Hn, X[:, :2], out=np.zeros_like(Hn), where=X[:, :2] > 1e-9)
            A = X[:, 2:4] * np.column_stack(
                [ratio[:, 0] ** model._p_of("blue", u), ratio[:, 1] ** model._p_of("red", u)]
            )
            X = np.column_stack([Hn, A, np.maximum(X[:, 4] + DT * theta @ model.coef["distance"], 0)])
        return X[:, 0] - X[:, 1]

    return predict


sources = ["hand-fitted reach curve", *feature_sets("blue"), "real share in reach"]
pd.DataFrame({s: compare(share_rollout(s)) for s in sources}).T.round(3)

**What the numbers say:** fill in. Two questions: how much better than today's numbers are the
new formation numbers, and how far does even the real share in reach get?

## 5 · Who dies when, and damage per unit in reach

Two more suspects for the rest of the hold advantage:

- **Who dies when:** at the same health lost, does a side lose more *units* under one intent?
  (The model ties deaths to health through one exponent, so it couldn't see that.)
- **Damage per unit in reach:** does a unit in reach do the same damage under both intents?
  (E.g. a walking Red might shoot less.)

In [ ]:
records = []
for (seed, movement), g in pairs.groupby(["seed", "movement"]):
    g = g.reset_index(drop=True)
    k = np.flatnonzero((g["reach_blue"] + g["reach_red"]) > 0)[0]
    rec = {
        "seed": seed,
        "movement": movement,
        "final margin": g["health_blue"].iloc[-1] - g["health_red"].iloc[-1],
    }
    for side, opp in (("red", "blue"), ("blue", "red")):
        lost = 1 - g[f"health_{side}"] / g[f"health_{side}"].iloc[0]
        dead = 1 - g[f"alive_{side}"] / g[f"alive_{side}"].iloc[0]
        for level in (0.1, 0.3):
            rec[f"{side}: share of units dead at {level:.0%} health lost"] = dead.iloc[
                np.flatnonzero(lost >= level)[0]
            ]
        third = np.flatnonzero(g[f"alive_{side}"] < g[f"alive_{side}"].iloc[0] - 0.3 + 1e-9)
        rec[f"{side}: third death, s after contact"] = (
            g["t"].iloc[third[0]] - g["t"].iloc[k] if len(third) else np.nan
        )
        w = g.iloc[k : k + 33]
        unit_seconds = 10 * w[f"reach_{opp}"].iloc[:-1].sum() * DT
        dealt = w[f"health_{side}"].iloc[0] - w[f"health_{side}"].iloc[-1]
        rec[f"{opp}: damage per unit in reach, first 4 s"] = dealt / max(unit_seconds / 10, 1e-9)
        rec[f"{opp}: unit-seconds in reach, first 4 s"] = unit_seconds
    records.append(rec)
deaths = pd.DataFrame(records)
h, a = (
    deaths[deaths["movement"] == m].set_index("seed").drop(columns="movement") for m in ("hold", "advance")
)
d = h - a
cols = [c for c in d.columns if c != "final margin"]
pd.DataFrame(
    {
        "hold": h[cols].mean(),
        "advance": a[cols].mean(),
        "hold - advance": d[cols].mean(),
        "corr. with the difference in final margin": d[cols].corrwith(d["final margin"]),
    }
).round(3)

**What the numbers say:** fill in. Is "who dies when" different between the intents? Is the
damage per unit in reach? What *is* different, and how strongly does it go with the outcome?

## 6 · A reach curve derived from the geometry

Section 4 showed the hand-fitted reach curve is the weak link. A curve derived from the game's
rule instead: a unit is in reach when the enemy's front unit is within the reach ρ (5.75, the
game's value, not fitted). Own units spread around their centre (std = own depth); the enemy's
front unit stands about β · enemy depth · m(n) ahead of the enemy centre, with
m(n) = √(2 ln n) the typical maximum of n spread positions:

s = Φ( (ρ − d_eff + β · depth_opp · m(n_opp)) / (g · depth_own) ),   d_eff = √(d² + λ · (width_own − width_opp)²)

(`intent_model.derived_share`). Fitted: λ (width effect), β (how far the front really stands
ahead), g (how gradual the switch is). Variants with fewer parameters, and one with ρ free as a check.

In [ ]:
from scipy.optimize import curve_fit  # noqa: E402

from sindy_wm.models.intent_model import derived_share  # noqa: E402


def derived_inputs(df, side):
    o = "red" if side == "blue" else "blue"
    cols = ["distance", f"depth_{side}", f"depth_{o}", f"width_{side}", f"width_{o}"]
    return (*(df[c].to_numpy() for c in cols), 10 * df[f"alive_{o}"].to_numpy())


variants = {
    "λ": (lambda X, lam: derived_share(*X, lam, 1, 1), [1.0]),
    "λ, β": (lambda X, lam, b: derived_share(*X, lam, b, 1), [1.0, 1.0]),
    "λ, β, g": (lambda X, lam, b, g: derived_share(*X, lam, b, g), [1.0, 1.0, 1.0]),
    "λ, β, g, ρ (check)": (
        lambda X, lam, b, g, rho: derived_share(*X, lam, b, g, rho),
        [1.0, 1.0, 1.0, 5.75],
    ),
}
fit_sample = f_train.sample(150_000, random_state=0)
derived_fits, rows_ = {}, []
for side in ("blue", "red"):
    rows_.append(
        {
            "side": side,
            "curve": "hand-fitted (6 parameters)",
            "R²": fit_table[("hand-fitted reach curve", side)],
        }
    )
    for name, (f, p0) in variants.items():
        p, _ = curve_fit(
            f,
            derived_inputs(fit_sample, side),
            fit_sample[f"reach_{side}"] / fit_sample[f"alive_{side}"],
            p0=p0,
            maxfev=20000,
        )
        derived_fits[(side, name)] = (f, p)
        pred = f_val[f"alive_{side}"].values * f(derived_inputs(f_val, side), *p)
        rows_.append(
            {
                "side": side,
                "curve": f"derived, fitted {name}",
                "R²": r2(f_val[f"reach_{side}"].values, pred),
                "parameters": np.round(p, 2),
            }
        )
    rows_.append(
        {
            "side": side,
            "curve": "flexible fit of depth, width (ceiling)",
            "R²": fit_table[("A depth, width", side)],
        }
    )
pd.DataFrame(rows_).set_index(["side", "curve"]).round(3)

The decision test from section 4, with the derived curve (real distance and shape, the model's
own units alive):

In [ ]:
def derived_rollout(name):
    def predict(movement):
        g = both[both["movement"] == movement]
        geo = np.full((len(seeds), H + 1, 5), np.nan)
        for j, (_, e) in enumerate(g.groupby("seed")):
            n = min(len(e), H + 1)
            geo[j, :n] = e[["distance", *SHAPE]].to_numpy()[:n]
        X, U = X0.astype(float).copy(), controls(movement)
        Ch = np.stack([model.coef["health_blue"], model.coef["health_red"]])
        for n in range(H):
            u = U[:, n]
            d, db, dr, wb, wr = _carry(geo, n).T
            fb, pb = derived_fits[("blue", name)]
            fr, pr = derived_fits[("red", name)]
            s = np.column_stack(
                [fb((d, db, dr, wb, wr, 10 * X[:, 3]), *pb), fr((d, dr, db, wr, wb, 10 * X[:, 2]), *pr)]
            )
            R = X[:, 2:4] * np.nan_to_num(s)
            F = library(X, u, S0, model.P, R, model.ctrl)
            theta = np.column_stack([F[k] for k in model.names])
            Hn = np.maximum(X[:, :2] + DT * theta @ Ch.T, 0)
            ratio = np.divide(Hn, X[:, :2], out=np.zeros_like(Hn), where=X[:, :2] > 1e-9)
            A = X[:, 2:4] * np.column_stack(
                [ratio[:, 0] ** model._p_of("blue", u), ratio[:, 1] ** model._p_of("red", u)]
            )
            X = np.column_stack([Hn, A, np.maximum(X[:, 4] + DT * theta @ model.coef["distance"], 0)])
        return X[:, 0] - X[:, 1]

    return predict


pd.DataFrame(
    {
        "hand-fitted reach curve": compare(share_rollout("hand-fitted reach curve")),
        "derived, fitted λ, β": compare(derived_rollout("λ, β")),
        "derived, fitted λ, β, g": compare(derived_rollout("λ, β, g")),
        "real share in reach": compare(share_rollout("real share in reach")),
    }
).T.round(3)

**In the full model and the planner** (computed outside this notebook: `model_derived.pkl`,
fitted like `model_split.pkl` but with the derived curve; `planner_derived.csv`, 400 battles):

| | hand-fitted curve | derived curve |
|---|---|---|
| health error at 4 s / 8 s (exp06 validation) | 0.057 / 0.103 | 0.057 / 0.099 |
| planner wins (always hold/focus 70.2%, advance/focus 67.0%) | 67.2% | **69.8%** |
| planner chooses hold / advance / withdraw | 48% / 52% / 1% | 51% / 45% / 5% |
| stalkers, health at 4 s: marine model scaled / refit | 0.036 / 0.024 | 0.040 / 0.024 |

Derived-curve planner against the hand-curve planner: +2.5 points (95% interval +0.2 to +4.8);
against always hold/focus: -0.5 (-4.0 to +3.2). Fitted separately on stalkers, β is 0.61 / 0.53
(marines 0.69 / 0.54).

**What the numbers say:** fill in.

## Read with care

- One pair of intents (hold/focus vs advance/focus), one scenario, battles that start close
  (contact after 0.4-0.7 s on average): there is little time for waiting to matter.
- The hold advantage is small: 3 points of win rate, 0.023 of final margin on average; the
  intents end the same in about half of the starts.
- Step 3 tried one library family. A different reach equation could do better; the derivative of
  reach is a staircase (one unit at a time), which makes its equation hard to fit (R² about 0.2).

## Questions for you

1. Section 5 says the hold advantage is units in reach in the first seconds, with the same damage
   per unit and the same deaths per health lost. Why does even the real share in reach (section 4)
   predict the outcome difference only partly? Think about how big the difference is (0.023)
   next to the model's error at 12 s.
2. Which kind of state could predict reach: groups (front and rear), or unit positions moved
   forward with simple kinematics? What would each cost in interpretability?
3. Is a 3-point effect worth modelling at all, or is the honest conclusion that hold and advance
   are nearly equivalent in this scenario?